In [1]:
# %pip install Bio

from KEGGRESTpy import kegg_get, kegg_link, kegg_find, kegg_list
from pprint import pprint
import random
import pandas as pd
import requests
import re
from Bio.KEGG import REST
import json, csv
import time
from pathlib import Path

In [2]:
# CODES
#  '00002', '00003',
code_database_k = ['00001', '01000', '01001','01009', '01002', '01003', '01005', '01011', '01004', '01008', '01006', '01007', '00199', '00194',
'03000', '03021', '03019', '03041', '03011', '03009', '03016', '03012', '03110', '04131', '04121', '03051', '03032', '03036', '03400', '03029',
'02000', '02044', '02042', '02022', '02035', '03037', '04812', '04147', '02048', '04030', '04050', '04054', '03310', '04040', '04031', '04052', '04515', '04090', '01504', '00535', '00536', '00537', '04091', '04990',
'03200', '03210',
'03100']

In [3]:
# CODE NAMES
name_database_k = ['KEGG Orthology (KO)',
'Enzymes', 'Protein kinases', 'Protein phosphatases and associated proteins', 'Peptidases and inhibitors', 'Glycosyltransferases', 'Lipopolysaccharide biosynthesis proteins', 'Peptidoglycan biosynthesis and degradation proteins', 'Lipid biosynthesis proteins', 'Polyketide biosynthesis proteins', 'Prenyltransferases', 'Amino acid related enzymes', 'Cytochrome P450', 'Photosynthesis proteins',
'Transcription factors', 'Transcription machinery', 'Messenger RNA biogenesis', 'Spliceosome', 'Ribosome', 'Ribosome biogenesis', 'Transfer RNA biogenesis', 'Translation factors', 'Chaperones and folding catalysts', 'Membrane trafficking', 'Ubiquitin system', 'Proteasome', 'DNA replication proteins', 'Chromosome and associated proteins', 'DNA repair and recombination proteins', 'Mitochondrial biogenesis',
'Transporters', 'Secretion system', 'Bacterial toxins', 'Two-component system', 'Bacterial motility proteins', 'Cilium and associated proteins', 'Cytoskeleton proteins', 'Exosome', 'Prokaryotic defense system', 'G protein-coupled receptors', 'Cytokine receptors', 'Pattern recognition receptors', 'Nuclear receptors', 'Ion channels', 'GTP-binding proteins', 'Cytokines and neuropeptides', 'Cell adhesion molecules', 'CD molecules', 'Antimicrobial resistance genes', 'Proteoglycans', 'Glycosaminoglycan binding proteins', 'Glycosylphosphatidylinositol (GPI)-anchored proteins', 'Lectins', 'Domain-containing proteins not elsewhere classified', 
'Viral proteins', 'Viral fusion proteins', 
'Non-coding RNAs']


In [ ]:
# read in keggs gene_product for all samples
# keggdb = {}
# with open(r"C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples\gene_product_kegg.json", "r") as f:
#     keggdb = json.load(f)
# # keggdb

def read_ko(directorypath: Path, filename: str, sample: str, keggdb: dict, genes: int):
    # Direct path, not rglob -- there's exactly one
    # real file per sample (confirmed live against all 128 samples), so
    # searching the whole subtree for matches only risks silently reading
    # a stray duplicate twice. Matches EGC_TAX_TREE.ipynb's read_tax.
    file_path = directorypath / filename
    if not file_path.is_file():
        print(f"No {filename} found in {directorypath}")
        return

    # No try/except here -- a real read
    # failure should crash the notebook loudly, not print a message that
    # scrolls past while the rest of the run continues on partial data.
    with file_path.open() as f:
        content = f.readlines()
    for line in content:
        info = line.strip().split('\t')
        if len(info) < 3:
            continue

        gene = info[0]
        keggid = info[2]

        if gene not in keggdb:
            keggdb[gene] = {'lookupId': [keggid], 'timepoint': [sample]}
        else:
            keggdb[gene]['lookupId'].append(keggid)
            keggdb[gene]['timepoint'].append(sample)

# Main code
with open(r'C:\Users\sakoehn\Desktop\JGI\sample_names.txt') as f:
    samples = [sample.strip() for sample in f]

base = Path(r"C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples")
keggdb = {}
genes = 0  # List so function can modify it

for s in samples:
    directpath = base / s
    filename = f"{s}.a.ko.txt"
    read_ko(directpath, filename, s, keggdb, genes)

print(f"Total unique genes: {genes}")
print(f"Total entries in keggdb: {len(keggdb)}")

In [5]:
keggdb['Ga0590986_0000001_9511_10863']

{'lookupId': ['KO:K00262'], 'timepoint': ['3300060358']}

In [6]:
# each node is a different element in KEGG
# keep track of the name/label, the KO id number, the direct parent, and the elements direct children
class Node:
    def __init__(self, label, ko=None, parent=None):
        self.label = label
        self.ko = ko
        self.parent = parent
        self.children = []
        self.timepoints = {}

# extracts what parent level the node is at, the ko id and the name of the particle
def get_info(parts):
    if len(parts) > 1:
        if parts[1][0:3] == '<b>':
            parts[1] = parts[1][3:]
       
        # if not bool(re.search(r'(\d.*){3,}', parts[1])) or bool(re.search(r'\d.', parts[1])): # no ko id number
            # print(parts)
        # print(parts[1])
        if not parts[1][1:].isnumeric(): 
            ko = " ".join(parts)[1:].lstrip()
            name = ko.lstrip()
            indent = parts[0][0]
        else:
            ko = parts[1]
            indent = parts[0]
            name = " ".join(parts[2:])
        
        if name[-4:] == '</b>': # string clean
            name = name[:-4]
        if ko[-4:] == '</b>':
            ko = ko[:-4]
            
        return [indent, ko, name]
    else:
        return None


# reads the txt to gather and store node information
def parse_brite(text, database_name, database_code):
    nodes = {}
    root = None
    parent_nodeA = None
    parent_nodeB = None
    parent_nodeC = None
    parent_nodeD = None
    children = []

    # cnt = 0
    for line in text.split('\n')[3:]:
        if not line.strip() or line.startswith('#'):
            continue
        if re.search(r'^A*', line):
            new_line = [line[0], line[1:]]
            line = " ".join(new_line)
        # print(line)
        parts = line.strip().split()
        info = get_info(parts)
        if info is None:
            continue
        root = Node(database_name, database_code) # gets root (aka category info)
        
        if info: # finds parent level 
            match info[0]:
                case 'A': 
                    # print(info[0], " ", info[1], " ", info[2])
                    current_node = Node(info[2], info[1], root) # create new element
                    current_node.category = f"{root.ko}  {root.label}" # keeps track of the category for future printing
                    root.children.append(f"{current_node.ko} {current_node.label}") # don't want the actual node, just the searchable info
                    parent_nodeA = current_node   # keep track of direct parent
                case 'B':
                    current_node = Node(info[2], info[1], parent_nodeA)
                    current_node.category = f"{root.ko}  {root.label}"
                    parent_nodeA.children.append(f"{current_node.ko} {current_node.label}")
                    parent_nodeB = current_node
                case 'C':
                    current_node = Node(info[2], info[1], parent_nodeB)
                    current_node.category = f"{root.ko}  {root.label}"
                    parent_nodeB.children.append(f"{current_node.ko} {current_node.label}")
                    parent_nodeC = current_node
                case 'D':
                    current_node = Node(info[2], info[1], parent_nodeC)
                    current_node.category = f"{root.ko}  {root.label}"
                    parent_nodeC.children.append(f"{current_node.ko} {current_node.label}")
                    parent_nodeD = current_node
                case 'E':
                    current_node = Node(info[2], info[1], parent_nodeD)
                    current_node.category = f"{root.ko}  {root.label}"
                    parent_nodeD.children.append(f"{current_node.ko} {current_node.label}")
            nodes[info[1]] = current_node
    return nodes

# takes each node infomration and creates a tree structure
def build_leaf_info(nodes):
    leaf_info = {}
    for node in nodes.values():
        if  node.ko and node.label:
            parent_name = node.parent.label if node.parent else None
            parent_ko = node.parent.ko if node.parent else None
        
            if parent_name and parent_ko:
                if parent_name == parent_ko:
                    parent = parent_name
                else:
                    parent = " ".join([parent_ko, parent_name])
            else:
                parent = None
                     
            leaf_info[node.ko] = {
                'name': node.label,
                'parent': parent,
                'children': node.children,
                'category': node.category,
                'timepoints': node.timepoints
            }
           
            leaf_info[node.label] = {
                'ko': node.ko,
                'parent': parent,
                'children': node.children,
                'category': node.category,
                'timepoints': node.timepoints
            }
    return leaf_info
    

In [7]:
# cnt = 0
# nodes = None
# for i in range(len(code_database_k)):
#     time.sleep(0.5)
#     url = f"https://www.kegg.jp/kegg-bin/download_htext?htext=ko{code_database_k[i]}.keg&format=htext"
#     response = requests.get(url).text
#     # print(response)
    
#     nodes = parse_brite(response, name_database_k[i], code_database_k[i])
#     # trees[code_database_k[i]] = build_leaf_info(nodes)
#     # trees[name_database_k[i]] = build_leaf_info(nodes)



# Pick ONE KEGG hierarchy (e.g., first one)
i = 0
url = f"https://www.kegg.jp/kegg-bin/download_htext?htext=ko{code_database_k[i]}.keg&format=htext"
response = requests.get(url).text

# This gives you ALL KEGG nodes with full hierarchy
nodes = parse_brite(response, name_database_k[i], code_database_k[i])

In [11]:
#['K00560']

{'Transfer RNA': <__main__.Node at 0x20913e9bbd0>,
 'K14218': <__main__.Node at 0x20913e9b950>,
 'K14219': <__main__.Node at 0x20913e9bed0>,
 'K14220': <__main__.Node at 0x20913e9b8d0>,
 'K14221': <__main__.Node at 0x20913e9bdd0>,
 'K14222': <__main__.Node at 0x20913e9ba50>,
 'K14223': <__main__.Node at 0x20913e9b850>,
 'K14224': <__main__.Node at 0x20913e9b750>,
 'K14225': <__main__.Node at 0x20913e9b650>,
 'K14226': <__main__.Node at 0x20913e9b9d0>,
 'K14227': <__main__.Node at 0x20913e9bb50>,
 'K14228': <__main__.Node at 0x208bb8db650>,
 'K14229': <__main__.Node at 0x208bb8db2d0>,
 'K14230': <__main__.Node at 0x208bb8d9e50>,
 'K14231': <__main__.Node at 0x208bb8db250>,
 'K14232': <__main__.Node at 0x208bb8da7d0>,
 'K14233': <__main__.Node at 0x208bb8dab50>,
 'K14234': <__main__.Node at 0x208bb8da9d0>,
 'K14235': <__main__.Node at 0x208bb8daad0>,
 'K14236': <__main__.Node at 0x208bb8dac50>,
 'K14237': <__main__.Node at 0x208bb8db450>,
 'K14238': <__main__.Node at 0x208bb8dbf50>,
 'K1

In [31]:
# for g in keggdb.keys(): # check to see if a gene has more than one ko
#     if len(keggdb[g]['lookupId']) > 1:
#         print(" multiple kos for gene: ", g, "  k: ", keggdb[g]['lookupId'])

for gene, ginfo in keggdb.items():
    if 'timepoints' not in ginfo:
        print(f"Missing timepoints for gene {gene}: {ginfo}")
        break

Missing timepoints for gene Ga0590986_0000001_3_974: {'lookupId': ['KO:K09014'], 'timepoint': ['3300060358']}


In [65]:
# for t, tree in trees.items():
#     for gene, ginfo in keggdb.items():
#         scaffold = gene.rsplit('_', 2)[0]
        
#         if not ginfo['lookupId'] or 'timepoints' not in ginfo:
#             continue

            
#         for ko_full in ginfo['lookupId']:
#             ko = ko_full[3:] 
#             node = tree.get(ko)
    
#             if node is None:
#                 continue

#             # For each sample this gene appears in
#             for sample in ginfo['timepoint']:

#                 if sample not in node['timepoints']:
#                     node['timepoints'][sample] = {'egc': 0, 'scaffolds': {}}
                    
#                     node['timepoints'][sample]['scaffolds'][scaffold] = 1
#                 # Add gene to this sample's list
#                 elif scaffold :
#                      node['timepoints'][sample]['scaffolds'][scaffold] += 1
#             # node['timepoints'][sample]['egc'] = 0  # Count genes for this KO in this sample

for t, tree in trees.items():  # tree = {ko: Node}
    for gene, ginfo in keggdb.items():
        scaffold = gene.rsplit('_', 2)[0]
        
        lookup_ids = ginfo.get('lookupId')
        timepoints = ginfo.get('timepoints')
        if not lookup_ids or not timepoints:
            continue

        for ko_full in lookup_ids:
            ko = ko_full[3:]
            node = tree.get(ko)  # Returns Node instance!
            if node is None:
                continue

            tp = node.timepoints  # ✅ Works - Node attribute

            for sample in timepoints:
                if sample not in tp:
                    tp[sample] = {'egc': 0, 'scaffolds': {}, 'genes': []}

                entry = tp[sample]
                entry['scaffolds'][scaffold] = entry['scaffolds'].get(scaffold, 0) + 1
                entry['egc'] += 1
                entry['genes'].append(gene)


In [80]:
trees

{}

In [28]:
# gets information (ko/name, parent, children) for specific kegg element
def acquire_info(koid, showgenes):
    result = ""
    fnd = []
    name = ""
    for cat in trees.keys():
        if koid in trees[cat].keys():
            if trees[cat][koid] not in fnd:
                fnd.append(trees[cat][koid])
    if not showgenes:
        for i in fnd:
            result += f"""
        Category: {i['category']}
             direct parent: {i['parent']}
             direct children: {i['children']}
             estimated gene copies: {i['copies']}
             genes hidden
    """
    else: 
        for i in fnd:
            result += f"""
        Category: {i['category']}
             direct parent: {i['parent']}
             direct children: {i['children']}
             estimated gene copies: {i['copies']}
             genes: {i['genes']}
    """
    if fnd != []:
        if koid[1:].isdigit():
            res = f"""{koid} {fnd[0]['name']}"""
        else:
            res = f"""{koid}"""
    else:
        res = f"""{koid}  ~  NOT FOUND"""

    res += result
    return res

In [1]:
result = acquire_info("K09014", True)


NameError: name 'acquire_info' is not defined

In [30]:
# MAKE SEARCH FUNCTION MORE GENERAL
# returns 
def general_search(term):
    # print(term)
    categories = {}
    fnd = 0
    for cat in list(trees.keys())[::2]:
        # if term in trees[cat].keys():
            for i in trees[cat]:
                
                if term.lower() in i.lower():
                    key = trees[cat][i]['category']
                    fnd += 1
                    # Initialize list if key not yet in categories
                    if key not in categories:
                        categories[key] = []
                    
                    if trees[cat][i]['ko'] == i:
                        categories[key].append(i)
                    else:
                        categories[key].append(trees[cat][i]['ko'] + " " + i)

    if fnd == 0:
        return f'{term} NOT FOUND.'

    result = f'''{term} ~ FOUND IN {fnd} ANNOTATIONS
    '''
    for key in categories.keys():
        if categories[key] != None:
            result += f'''
    category: {key}'''
            for i in categories[key]:
                result += f'''
                {i}'''
    return result


In [2]:
result = general_search(" iron ")


NameError: name 'general_search' is not defined

In [3]:
result = acquire_info("K03783", False)


NameError: name 'acquire_info' is not defined

In [16]:
for gene in keggdb.keys():
    if keggdb[gene]['lookupId'] == 'K02677':
        print("Error: K02677 exists but is not found in tree.")

In [17]:
# want number of copies of genes that were found. 

# type in iron: returns the estimated gene copy number as a function of time. 
#  classical protein kinase C alpha type [EC:2.7.11.13] : gene count and time (abundance of genes at each time point)
# all different genes that have cnt associated with them summed. 
# for given ko, want total estiamted gene copies/count for every gene found that is associated with the ko number



# a.faa = amino acid protein database (fasta file)
# get dna samples jgi 


In [33]:
# gene to KO id numbers
gene_data = {}
for gene in keggdb.keys():
    if gene not in keggdb.keys():
        gene_data[gene] = []
    ko = genes[gene]
    if ko not in gene_data[gene]:
        gene_data[gene].append(ko)
for g in gene_data.keys():
    gene_data[g].sort()
# gene_data

NameError: name 'genes' is not defined

In [34]:
def gene_search(gene):
    ko = gene_data[gene]
    result = acquire_info(ko[0])
    return result

In [35]:
result = gene_search('Ga0590984_0000001_574_1893')
print(result)

KeyError: 'Ga0590984_0000001_574_1893'